In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Silver_Ingestion").getOrCreate()
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import logging

In [0]:
bronze_table = 'rwd.bronze.encounter'
SILVER_TABLE      = "rwd.silver.encounter"
silver_checkpoint = "/Volumes/rwd/bronze/myvolume/bronze/encounter_raw/silver"
BUSINESS_KEY="patient_id"
TRACKED_COLUMNS   = ["Encounter_ID", "Patient_ID", "Cancer_Disease_Status", "Last_Update", "On_Treatment"]

In [0]:
def setup_tables():
 
    
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {SILVER_TABLE} (
            Encounter_ID              STRING       ,
            Patient_ID                 STRING,
            Cancer_Disease_Status       STRING,
            Last_Update                 DATE,
            On_Treatment                STRING,
            scd_start_date          TIMESTAMP    NOT NULL,
            scd_end_date            TIMESTAMP,
            is_current              BOOLEAN      NOT NULL,
            scd_version             BIGINT       NOT NULL,
            _bronze_load_datetime   TIMESTAMP,
            _silver_load_datetime   TIMESTAMP
        )
        USING DELTA
        TBLPROPERTIES (
            'delta.enableChangeDataFeed'       = 'true',
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact'   = 'true'
        )
    """)


In [0]:
# Call setup_tables to create the Silver table
#setup_tables()

In [0]:
def cast_columns():
    """Cast raw Bronze strings to typed Silver columns."""
    df=spark.read.table("rwd.bronze.encounter")
    return (
        df
        .withColumn("Last_Update",            F.to_date(F.col("Last_Update"), "yyyy-MM-dd"))
        .withColumn("Encounter_ID", F.trim(F.col("Encounter_ID")))
        .withColumn("Cancer_Disease_Status",           F.trim(F.col("Cancer_Disease_Status")))
        .withColumn("On_Treatment",         F.upper(F.trim(F.col("On_Treatment"))))
        .withColumn("Patient_ID",            F.trim(F.col("Patient_ID")))
    )

In [0]:
def apply_scd2(micro_batch_df, batch_id):
    now = F.current_timestamp()

    # Deduplicate within batch
    deduped = (
        micro_batch_df
        .withColumn(
            "rn",
            F.row_number().over(
                Window.partitionBy("Patient_ID")
                      .orderBy(F.col("_load_datetime").desc())
            )
        )
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

    deduped.createOrReplaceTempView("incoming_updates")

    # Proper SCD2 MERGE
    spark.sql(f"""
        MERGE INTO {SILVER_TABLE} AS tgt
        USING incoming_updates AS src
        ON tgt.Patient_ID = src.Patient_ID AND tgt.is_current = true

        -- Expire current row if something changed
        WHEN MATCHED AND (
            tgt.Encounter_ID <> src.Encounter_ID OR
            tgt.Cancer_Disease_Status <> src.Cancer_Disease_Status OR
            tgt.Last_Update <> src.Last_Update OR
            tgt.On_Treatment <> src.On_Treatment
        )
          THEN UPDATE SET tgt.is_current = false, tgt.scd_end_date = current_timestamp()

        -- Insert new row (either brand new patient or updated patient)
        WHEN NOT MATCHED
          THEN INSERT (
            Encounter_ID, Patient_ID, Cancer_Disease_Status, Last_Update, On_Treatment,
            scd_start_date, scd_end_date, is_current, scd_version,
            _bronze_load_datetime, _silver_load_datetime
          )
          VALUES (
            src.Encounter_ID, src.Patient_ID, src.Cancer_Disease_Status, src.Last_Update, src.On_Treatment,
            current_timestamp(), null, true,
            COALESCE(tgt.scd_version + 1, 1),   -- increment version if exists, else start at 1
            src._load_datetime, current_timestamp()
          )
    """)


In [0]:
def ingest_silver():
 

    cdf_stream = (
        spark.readStream
             .format("delta")
             .option("readChangeFeed", "true")
             .option("startingVersion", 0)   # use 0 or a valid version number
             .table(bronze_table)
             .filter(F.col("_change_type").isin("insert", "update_postimage"))
    )

    query = (
        cdf_stream.writeStream
                  .foreachBatch(apply_scd2)   # your SCD2 merge function
                  .option("checkpointLocation", silver_checkpoint)
                  .trigger(availableNow=True)
                  .start()
    )

    query.awaitTermination()


In [0]:
ingest_silver()

In [0]:
%sql

select * from rwd.silver.encounter;

Encounter_ID,Patient_ID,Cancer_Disease_Status,Last_Update,On_Treatment,scd_start_date,scd_end_date,is_current,scd_version,_bronze_load_datetime,_silver_load_datetime
E005,P005,Active,2026-01-18,Yes,2026-03-17T17:50:44.905Z,null,true,1,2026-03-17T16:48:42.911Z,2026-03-17T17:50:44.905Z
E008,P008,Stable,2026-01-12,Yes,2026-03-17T17:50:44.905Z,null,true,1,2026-03-17T16:48:42.911Z,2026-03-17T17:50:44.905Z
E001,P001,Active,2026-01-15,Yes,2026-03-17T17:50:44.905Z,null,true,1,2026-03-17T16:48:42.911Z,2026-03-17T17:50:44.905Z
E006,P006,Remission,2025-11-30,No,2026-03-17T17:50:44.905Z,null,true,1,2026-03-17T16:48:42.911Z,2026-03-17T17:50:44.905Z
E002,P002,Remission,2025-12-20,No,2026-03-17T17:50:44.905Z,null,true,1,2026-03-17T16:48:42.911Z,2026-03-17T17:50:44.905Z
E003,P003,Progressive,2026-01-10,Yes,2026-03-17T17:50:44.905Z,null,true,1,2026-03-17T16:48:42.911Z,2026-03-17T17:50:44.905Z
E007,P007,Progressive,2026-01-22,Yes,2026-03-17T17:50:44.905Z,null,true,1,2026-03-17T16:48:42.911Z,2026-03-17T17:50:44.905Z
E004,P004,Stable,2026-01-05,Yes,2026-03-17T17:50:44.905Z,null,true,1,2026-03-17T16:48:42.911Z,2026-03-17T17:50:44.905Z
E010,P010,Remission,2025-12-25,No,2026-03-17T17:50:44.905Z,2026-03-17T18:16:19.532Z,false,1,2026-03-17T16:48:42.911Z,2026-03-17T17:50:44.905Z
E011,P011,Active,2024-10-12,No,2026-03-17T18:16:19.532Z,null,true,1,2026-03-17T18:03:20.092Z,2026-03-17T18:16:19.532Z
